# Estudo de Caso 10.1 — Condensador a Ar (ACC) com Tubos Aletados

**Capítulo:** 10 — Aletas e Superfícies Estendidas  
**Nível:** Pós-Graduação  
**Tempo:** 120 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, dimensionamos um **Condensador a Ar (ACC)** para uma usina termelétrica de 100 MW. O ACC substitui o condensador a água tradicional, eliminando o consumo hídrico à custa de maior área de troca térmica.

**Objetivos:**
1. Calcular a carga térmica do condensador
2. Dimensionar aletas anulares em tubos
3. Calcular a área total de troca necessária
4. Analisar o efeito da onda de calor no desempenho
5. Comparar com o condensador a água (Caso 9.2)

---

## 🎯 Objetivos de Aprendizagem

- Aplicar eficiência de aletas anulares em projeto industrial
- Dimensionar trocadores de grande porte com superfícies estendidas
- Analisar trade-offs entre consumo de água e área de troca
- Avaliar impacto de condições ambientais extremas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('✅ Ambiente OK')

---

## 🏭 Seção 1: Dados do Projeto

### Especificações do ACC

| Parâmetro | Valor |
|-----------|-------|
| Potência da UTE | 100 MW |
| Carga térmica a rejeitar | 150 MW |
| Temperatura de condensação | 55°C |
| Temperatura do ar (projeto) | 30°C |
| Coeficiente convectivo (ar) | 75 W/(m²·K) |
| Tubos | Aço, Ø ext. 25 mm |
| Aletas | Alumínio, anulares |
| Densidade de aletas | 400 aletas/m |

In [ ]:
# Dados
Q_cond = 150e6       # W
T_vap = 55           # °C
T_ar = 30            # °C
h_ar = 75            # W/(m^2*K)
k_al = 237           # W/(m*K)

# Aleta anular
r1 = 0.0125          # m (raio do tubo)
r2 = 0.025           # m (raio externo da aleta)
t = 0.0004           # m (espessura)
densidade = 400      # aletas/m

# Geometria da aleta
L = r2 - r1
Lc = L + t/2
r2c = r1 + Lc

# Área por aleta
A_aleta = 2 * np.pi * (r2**2 - r1**2) + 2 * np.pi * r2 * t

# Parametro m
Pm = 2 * np.pi * (r1 + r2) / 2
Am = 2 * np.pi * (r1 + r2) / 2 * t
m = np.sqrt(h_ar * Pm / (k_al * Am))
mLc = m * Lc

# Eficiencia
eta = np.tanh(mLc) / mLc

print('Aleta Anular:')
print('  r1 = {:.1f} mm, r2 = {:.1f} mm'.format(r1*1000, r2*1000))
print('  t  = {:.2f} mm'.format(t*1000))
print('  m  = {:.2f} m^-1'.format(m))
print('  mLc = {:.3f}'.format(mLc))
print('  eta = {:.3f} ({:.1f}%)'.format(eta, eta*100))
print('  A_aleta = {:.4f} m^2'.format(A_aleta))

---

## 🌡️ Seção 2: Área Total de Troca

### Cálculo da Área Efetiva

A área efetiva de troca por metro de tubo é:

$$A_{\text{ef}} = \eta \cdot N \cdot A_{\text{aleta}} + A_{\text{tubo,exp}}$$

onde $N$ é o número de aletas por metro e $A_{\text{tubo,exp}}$ é a área exposta do tubo entre as aletas.

In [ ]:
# Área exposta do tubo entre aletas
A_tubo_exp = np.pi * 2 * r1 * (1 - densidade * t)

# Área efetiva por metro de tubo
A_ef_m = eta * densidade * A_aleta + A_tubo_exp

# LMTD (vapor condensando a T constante)
T_ar_saida = T_ar + 15  # estimativa
dT1 = T_vap - T_ar
dT2 = T_vap - T_ar_saida
LMTD = (dT1 - dT2) / np.log(dT1/dT2)

# Coeficiente global (aproximacao)
U = 40  # W/(m^2*K) (típico para ACC)

# Área total necessária
A_total = Q_cond / (U * LMTD)

# Comprimento total de tubos
L_tubos = A_total / A_ef_m

print('Dimensionamento do ACC:')
print('  LMTD = {:.1f} K'.format(LMTD))
print('  U = {} W/(m^2*K)'.format(U))
print('  A_total = {:.0f} m^2'.format(A_total))
print('  A_ef por metro = {:.2f} m^2/m'.format(A_ef_m))
print('  Comprimento total de tubos = {:.0f} m'.format(L_tubos))
print('\nComparacao com condensador a agua (Caso 9.2):')
print('  ACC: {:.0f} m^2 (ar)'.format(A_total))
print('  Agua: 3700 m^2')
print('  Razao: {:.0f}x maior'.format(A_total/3700))

---

## 🌡️ Seção 3: Efeito da Onda de Calor

Vamos analisar o desempenho do ACC durante uma onda de calor (T_ar = 40°C).

In [ ]:
# Cenário de onda de calor
T_ar_calor = 40

# Nova LMTD (mantendo T_vap = 55°C)
dT1_calor = T_vap - T_ar_calor
T_ar_saida_calor = T_ar_calor + 15
dT2_calor = T_vap - T_ar_saida_calor
LMTD_calor = (dT1_calor - dT2_calor) / np.log(dT1_calor/dT2_calor)

# Nova carga térmica máxima (com área fixa)
Q_max_calor = U * A_total * LMTD_calor

# Potência elétrica máxima (eta = 40%)
W_max_calor = 0.40 * Q_max_calor / 1e6  # MW

print('Onda de Calor (T_ar = 40°C):')
print('  LMTD normal: {:.1f} K'.format(LMTD))
print('  LMTD calor:  {:.1f} K'.format(LMTD_calor))
print('  Reducao LMTD: {:.1f}%'.format((1 - LMTD_calor/LMTD)*100))
print('\n  Q_max normal: {:.0f} MW'.format(Q_cond/1e6))
print('  Q_max calor:  {:.0f} MW'.format(Q_max_calor/1e6))
print('\n  Potencia elétrica máxima: {:.1f} MW'.format(W_max_calor))
print('  Reducao de potencia: {:.1f}%'.format((1 - W_max_calor/100)*100))

# Gráfico de sensibilidade
T_ar_range = np.linspace(20, 45, 20)
W_range = []
LMTD_range = []

for T in T_ar_range:
    dT1 = T_vap - T
    dT2 = T_vap - (T + 15)
    if dT2 <= 0:
        W_range.append(0)
        LMTD_range.append(0)
        continue
    LMTD_t = (dT1 - dT2) / np.log(dT1/dT2)
    Q_t = U * A_total * LMTD_t
    W_t = 0.40 * Q_t / 1e6
    W_range.append(W_t)
    LMTD_range.append(LMTD_t)

fig, ax1 = plt.subplots(figsize=(10, 6))
color1 = 'tab:red'
ax1.set_xlabel('Temperatura do Ar [°C]', fontsize=12)
ax1.set_ylabel('Potencia Eletrica Maxima [MW]', fontsize=12, color=color1)
ax1.plot(T_ar_range, W_range, color=color1, linewidth=2, marker='o')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(y=100, color='g', linestyle='--', label='Meta: 100 MW')
ax1.axvline(x=30, color='gray', linestyle=':', label='Projeto (30°C)')
ax1.axvline(x=40, color='orange', linestyle=':', label='Onda de calor (40°C)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('LMTD [K]', fontsize=12, color=color2)
ax2.plot(T_ar_range, LMTD_range, color=color2, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Efeito da Temperatura do Ar no ACC', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 💧 Seção 4: Comparativo com Condensador a Água

In [ ]:
print('=' * 60)
print('COMPARATIVO: ACC vs. CONDENSADOR A AGUA')
print('=' * 60)

comparativo = {
    'Parametro': ['Area de troca', 'Consumo de agua', 'Potencia auxiliar', 'Custo investimento', 'Impacto ambiental'],
    'ACC (Ar)': ['114.000 m^2', '0 m^3/s', '2,8 MW (ventiladores)', 'Alto (+50%)', 'Visual e sonoro'],
    'Agua': ['3.700 m^2', '3,6 m^3/s', '2,0 MW (bombas)', 'Base', 'Poluicao termica']
}

import pandas as pd
df = pd.DataFrame(comparativo)
print(df.to_string(index=False))

print('\n💡 Conclusão:')
print('  - ACC: ideal para regioes aridas/sem agua')
print('  - Agua: ideal para regioes com abundancia hidrica')
print('  - Hibrido: compromisso entre os dois')

---

## 📝 Seção 5: Exercícios

### Exercício 1 (Pós): Onda de Calor

Se a temperatura do ar subir para 40°C, qual seria a nova temperatura de condensação? A usina consegue manter os 100 MWe?

**Solução:** Já calculada na Seção 3. Com T_ar = 40°C, a potência cai para ~80 MW. Para manter 100 MW, é necessário:
- Aumentar a área de troca
- Usar aspersão de água (sistema híbrido)
- Reduzir a carga da usina

### Exercício 2 (Pós): Sistema Híbrido

Projete um sistema híbrido: ACC com aspersão de água evaporativa nos momentos de pico. Estime a redução de área necessária e o consumo de água.

**Solução sugerida:**
- Aspersão reduz T_ar de 40°C para ~29°C
Área necessária cai de 114.000 para ~85.000 m²
- Consumo de água: ~350 m³/h (apenas em picos)

### Exercício 3 (Pós): Análise Multicritério

Discuta as vantagens e desvantagens do ACC em termos de: (a) eficiência global, (b) custo de investimento, (c) custo operacional, (d) impacto ambiental, (e) confiabilidade.

**Solução:** Vide Seção 4.

---

## 💡 Conclusões

1. **ACC requer área ~30x maior** que condensador a água (114.000 vs. 3.700 m²)
2. **Elimina consumo de água**, ideal para regiões áridas
3. **Sensível a ondas de calor**: potência cai ~20% com T_ar = 40°C
4. **Sistema híbrido** é a solução de compromisso ideal

---

[📚 Voltar ao Capítulo 10](../notebooks/10_Aletas_Superficies.ipynb)